In [8]:
from __future__ import annotations

import os
from typing import Optional, Tuple

from datasets import (
    Dataset,
    DatasetDict,
    IterableDataset,
    IterableDatasetDict,
    load_dataset,
)

SEED = 42
N_SAMPLES = 100

# Save sampled files directly under `benchmarks/`
OUT_DIR = "benchmarks"
os.makedirs(OUT_DIR, exist_ok=True)


def _pick_split(ds) -> Tuple[object, str]:
    """Pick a split to sample from, preferring benchmark-like splits."""
    preferred_splits = (
        "benchmark",
        "benchmarks",
        "train",
        "validation",
        "test",
    )

    # Map-style datasets
    if isinstance(ds, DatasetDict):
        for name in preferred_splits:
            if name in ds:
                return ds[name], name
        name = next(iter(ds.keys()))
        return ds[name], name

    # Iterable datasets
    if isinstance(ds, IterableDatasetDict):
        for name in preferred_splits:
            if name in ds:
                return ds[name], name
        name = next(iter(ds.keys()))
        return ds[name], name

    # Single-split dataset already
    return ds, "(single)"


def sample_dataset(
    hf_name: str,
    *,
    n: int = N_SAMPLES,
    seed: int = SEED,
    split: Optional[str] = None,
    streaming: bool = True,
) -> Dataset:
    """Load `hf_name`, take a reproducible random sample of `n`, and return it.

    Uses streaming first (to avoid full download) and falls back to non-streaming.
    """

    def _load(*, streaming_flag: bool):
        if split is None:
            return load_dataset(hf_name, streaming=streaming_flag)
        return load_dataset(hf_name, split=split, streaming=streaming_flag)

    ds = None
    if streaming:
        try:
            ds = _load(streaming_flag=True)
        except Exception:
            ds = None

    if ds is None:
        ds = _load(streaming_flag=False)

    chosen, chosen_name = _pick_split(ds)

    if isinstance(chosen, IterableDataset):
        # Shuffle within a buffer and take `n` items (does not require full download).
        sampled_iter = chosen.shuffle(seed=seed, buffer_size=max(10_000, n * 50)).take(n)
        sampled = Dataset.from_list(list(sampled_iter))
    else:
        n_eff = min(n, len(chosen))
        sampled = chosen.shuffle(seed=seed).select(range(n_eff))

    print(f"{hf_name}: sampled {len(sampled)} rows from split {chosen_name}")
    return sampled


def save_sample(sampled: Dataset, out_basename: str) -> str:
    out_path = os.path.join(OUT_DIR, f"{out_basename}.parquet")
    sampled.to_parquet(out_path, compression="snappy")
    return out_path


In [9]:
datasets_to_sample = {
    "GAIR/LIMR": "GAIR__LIMR",
    "Open-Reasoner-Zero/orz_math_72k_collection_extended": "Open-Reasoner-Zero__orz_math_72k_collection_extended",
    "PRIME-RL/Eurus-2-RL-Data": "PRIME-RL__Eurus-2-RL-Data",
}

sampled_paths = {}
for hf_name, out_base in datasets_to_sample.items():
    sampled = sample_dataset(hf_name, n=N_SAMPLES, seed=SEED)
    out_path = save_sample(sampled, out_base)
    sampled_paths[hf_name] = out_path

sampled_paths

Repo card metadata block was not found. Setting CardData to empty.


GAIR/LIMR: sampled 100 rows from split train


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 174.23ba/s]


Open-Reasoner-Zero/orz_math_72k_collection_extended: sampled 100 rows from split train


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1411.27ba/s]


PRIME-RL/Eurus-2-RL-Data: sampled 100 rows from split train


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1318.55ba/s]


{'GAIR/LIMR': 'benchmarks/GAIR__LIMR.parquet',
 'Open-Reasoner-Zero/orz_math_72k_collection_extended': 'benchmarks/Open-Reasoner-Zero__orz_math_72k_collection_extended.parquet',
 'PRIME-RL/Eurus-2-RL-Data': 'benchmarks/PRIME-RL__Eurus-2-RL-Data.parquet'}

In [10]:
# Quick peek at columns + 2 examples per dataset
for hf_name in datasets_to_sample.keys():
    ds = load_dataset(hf_name)
    split_ds, split_name = _pick_split(ds)

    if isinstance(split_ds, IterableDataset):
        preview = Dataset.from_list(list(split_ds.take(2)))
        cols = preview.column_names
    else:
        cols = split_ds.column_names
        preview = split_ds.select(range(min(2, len(split_ds))))

    print("\n---")
    print(hf_name)
    print("split:", split_name)
    print("columns:", cols)
    print(preview)

Repo card metadata block was not found. Setting CardData to empty.



---
GAIR/LIMR
split: train
columns: ['id', 'prompt', 'answer', 'source']
Dataset({
    features: ['id', 'prompt', 'answer', 'source'],
    num_rows: 2
})

---
Open-Reasoner-Zero/orz_math_72k_collection_extended
split: train
columns: ['0', '1']
Dataset({
    features: ['0', '1'],
    num_rows: 2
})


Generating validation split: 100%|██████████| 2048/2048 [00:00<00:00, 2706.81 examples/s]


---
PRIME-RL/Eurus-2-RL-Data
split: train
columns: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info']
Dataset({
    features: ['data_source', 'prompt', 'ability', 'reward_model', 'extra_info'],
    num_rows: 2
})


In [3]:
import os
import getpass
from huggingface_hub import HfApi, login

# --- Login (safe: reads token from env or prompt) ---
_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
if not _token:
    _token = getpass.getpass("Hugging Face token (will not echo): ")
login(token=_token)

api = HfApi()

private = (os.environ.get("HF_DATASET_PRIVATE", "0").strip() == "1")

uploads = [
    {
        "repo_id": "talzoomanzoo/limr",
        "local_path": "/scratch/mjgwak/rl-data-contamination-mj/benchmarks/LIMR/limr.parquet",
        "path_in_repo": "limr.parquet",
    },
    {
        "repo_id": "talzoomanzoo/eurus",
        "local_path": "/scratch/mjgwak/rl-data-contamination-mj/benchmarks/EURUS/eurus.parquet",
        "path_in_repo": "eurus.parquet",
    },
    {
        "repo_id": "talzoomanzoo/openreasoner",
        "local_path": "/scratch/mjgwak/rl-data-contamination-mj/benchmarks/OPENREASONER/openreasoner.parquet",
        "path_in_repo": "openreasoner.parquet",
    },
]

# --- Check files ---
missing = [u["local_path"] for u in uploads if not os.path.exists(u["local_path"])]
if missing:
    raise FileNotFoundError("Missing local files:\n" + "\n".join(missing))

print("Uploading datasets...")

for u in uploads:
    repo_id = u["repo_id"]
    local_path = u["local_path"]
    path_in_repo = u["path_in_repo"]

    # 1. Create dataset repo if it doesn't exist
    try:
        api.create_repo(
            repo_id=repo_id,
            repo_type="dataset",
            private=private,
            exist_ok=True,
        )
        print(f"Repo ready: {repo_id}")
    except Exception as e:
        print(f"Repo creation skipped/failed: {repo_id} ({e})")

    # 2. Upload file
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=path_in_repo,
        repo_id=repo_id,
        repo_type="dataset",
    )

    print(f"Uploaded: {local_path} -> {repo_id}/{path_in_repo}")

print("All uploads complete.")

Uploading datasets...
Repo ready: talzoomanzoo/limr


Processing Files (1 / 1): 100%|██████████| 25.7kB / 25.7kB, 11.7kB/s  
New Data Upload: 100%|██████████| 25.7kB / 25.7kB, 11.7kB/s  


Uploaded: /scratch/mjgwak/rl-data-contamination-mj/benchmarks/LIMR/limr.parquet -> talzoomanzoo/limr/limr.parquet
Repo ready: talzoomanzoo/eurus


Processing Files (1 / 1): 100%|██████████| 21.1kB / 21.1kB, 13.2kB/s  
New Data Upload: 100%|██████████| 21.1kB / 21.1kB, 13.2kB/s  


Uploaded: /scratch/mjgwak/rl-data-contamination-mj/benchmarks/EURUS/eurus.parquet -> talzoomanzoo/eurus/eurus.parquet
Repo ready: talzoomanzoo/openreasoner


Processing Files (1 / 1): 100%|██████████| 22.4kB / 22.4kB, 14.0kB/s  
New Data Upload: 100%|██████████| 22.4kB / 22.4kB, 14.0kB/s  


Uploaded: /scratch/mjgwak/rl-data-contamination-mj/benchmarks/OPENREASONER/openreasoner.parquet -> talzoomanzoo/openreasoner/openreasoner.parquet
All uploads complete.


In [ ]:
from datasets import load_dataset


## Download `talzoomanzoo/eurus_member` locally

This downloads the dataset from Hugging Face and writes it under `benchmarks/EURUS/`.


In [1]:
import os
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from datasets import DatasetDict, load_dataset

HF_REPO = "talzoomanzoo/eurus_member"

ds = load_dataset(HF_REPO)
if isinstance(ds, DatasetDict):
    split_name = "train" if "train" in ds else list(ds.keys())[0]
    d = ds[split_name]
else:
    split_name = "(single)"
    d = ds

# Write under `benchmarks/EURUS/` whether cwd is repo-root or `benchmarks/`
base = Path(".")
if (base / "benchmark.ipynb").exists():
    out_dir = base / "EURUS"
else:
    out_dir = base / "benchmarks" / "EURUS"

out_dir.mkdir(parents=True, exist_ok=True)

# Save as parquet (easy to reuse)
out_parquet = out_dir / "eurus_member.parquet"
try:
    d.to_parquet(str(out_parquet))
except Exception:
    d.to_pandas().to_parquet(out_parquet, index=False)

print(f"Loaded {HF_REPO} split={split_name} rows={len(d)}")
print(f"Wrote: {out_parquet}")

display(d.to_pandas().head(5))


/home/mjgwak/miniconda3/envs/uid/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 678.25ba/s]

Loaded talzoomanzoo/eurus_member split=train rows=60
Wrote: EURUS/eurus_member.parquet


,data_source,prompt,member
0,aime26,[{'content': ' When tackling complex reasoning...,0
1,aime26,[{'content': ' When tackling complex reasoning...,0
2,aime26,[{'content': ' When tackling complex reasoning...,0
3,aime26,[{'content': ' When tackling complex reasoning...,0
4,aime26,[{'content': ' When tackling complex reasoning...,0


In [1]:
import os
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from datasets import DatasetDict, load_dataset

HF_REPO = "talzoomanzoo/still_member"

ds = load_dataset(HF_REPO)
if isinstance(ds, DatasetDict):
    split_name = "train" if "train" in ds else list(ds.keys())[0]
    d = ds[split_name]
else:
    split_name = "(single)"
    d = ds

# Write under `benchmarks/EURUS/` whether cwd is repo-root or `benchmarks/`
base = Path(".")
if (base / "benchmark.ipynb").exists():
    out_dir = base / "STILL"
else:
    out_dir = base / "benchmarks" / "STILL"

out_dir.mkdir(parents=True, exist_ok=True)

# Save as parquet (easy to reuse)
out_parquet = out_dir / "still_member.parquet"
try:
    d.to_parquet(str(out_parquet))
except Exception:
    d.to_pandas().to_parquet(out_parquet, index=False)

print(f"Loaded {HF_REPO} split={split_name} rows={len(d)}")
print(f"Wrote: {out_parquet}")

display(d.to_pandas().head(5))


/home/mjgwak/miniconda3/envs/uid/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1316.07ba/s]

Loaded talzoomanzoo/still_member split=train rows=90
Wrote: STILL/still_member.parquet


,data_source,prompt,member
0,aime26,[{'content': ' When tackling complex reasoning...,0
1,aime26,[{'content': ' When tackling complex reasoning...,0
2,aime26,[{'content': ' When tackling complex reasoning...,0
3,aime26,[{'content': ' When tackling complex reasoning...,0
4,aime26,[{'content': ' When tackling complex reasoning...,0
